# 18 - EASA Release Gate

Evaluates an exact report version against approved inventory, all six quality dimensions, quarantine, exceptions, and human approval. It records readiness evidence only; it never calls an authority endpoint or auto-submits a report.

In [ ]:
# PARAMETERS - supplied only after compliance-owner review.
environment_name = 'dev'
submission_id = ''
action = 'EXPORT'
actor_object_id = ''
config_root = '/lakehouse/default/Files/easa/config'
evidence_root = '/lakehouse/default/Files/easa/evidence'

import hashlib
import json
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

action = action.upper()
assert environment_name in {'dev', 'test', 'prod'}
assert action in {'EXPORT', 'TRANSMIT'}
assert submission_id, 'submission_id is required'
assert actor_object_id, 'actor_object_id is required'

def read_json(path):
    return json.loads(notebookutils.fs.head(path, 4 * 1024 * 1024))

matrix = read_json(config_root + '/easa_requirements_matrix.json')
deployment = read_json(config_root + '/easa_deployment.json')
environment = deployment['environments'][environment_name]

In [ ]:
submission_rows = spark.table('gold_easa_submission_snapshot').filter(F.col('submission_id') == submission_id).orderBy(F.col('submission_version').desc()).limit(1).collect()
assert len(submission_rows) == 1, 'BLOCKED_SUBMISSION_NOT_FOUND'
submission = submission_rows[0].asDict()
requirements = [item for item in matrix['requirements'] if item['requirement_id'] == submission['requirement_id']]
assert len(requirements) == 1, 'BLOCKED_REQUIREMENT_NOT_FOUND'
requirement = requirements[0]
blockers = []

if not requirement['inventory_approved'] or requirement['approval_status'] != 'APPROVED':
    blockers.append('REQUIREMENT_NOT_APPROVED')
if requirement['automation_eligibility'] != 'ELIGIBLE':
    blockers.append('AUTOMATION_NOT_ELIGIBLE')
if 'TODO' in json.dumps({key: requirement[key] for key in ['regulation', 'submission', 'authority', 'deadline', 'source_fields', 'validation_rules', 'output_format']}).upper():
    blockers.append('UNVERIFIED_REQUIREMENT_OR_TEMPLATE')

required_dimensions = {'RECONCILIATION', 'COMPLETENESS', 'VALIDITY', 'DUPLICATE', 'TIMELINESS', 'CROSS_FIELD'}
quality = spark.table('gold_easa_quality_result').filter(F.col('requirement_id') == requirement['requirement_id'])
latest_run = quality.agg(F.max('validation_run_id').alias('run_id')).collect()[0]['run_id']
if not latest_run:
    blockers.append('QUALITY_EVIDENCE_MISSING')
else:
    latest_quality = quality.filter(F.col('validation_run_id') == latest_run)
    passed_dimensions = {row['quality_dimension'] for row in latest_quality.filter((F.col('severity') == 'BLOCKING') & (F.col('result_status') == 'PASS')).select('quality_dimension').distinct().collect()}
    for dimension in sorted(required_dimensions - passed_dimensions):
        blockers.append('QUALITY_' + dimension + '_NOT_PASS')

quarantine_count = spark.table('silver_easa_quarantine').filter(
    (F.col('requirement_id') == requirement['requirement_id'])
    & (F.col('airport_id') == submission['airport_id'])
    & F.col('resolved_at_utc').isNull()).count()
if quarantine_count:
    blockers.append('UNRESOLVED_QUARANTINE')

approvals = spark.table('gold_easa_approval_event').filter(
    (F.col('submission_id') == submission_id)
    & (F.col('report_version_hash') == submission['report_version_hash'])
    & (F.col('approval_status') == 'APPROVED')).orderBy(F.col('decision_at_utc').desc()).limit(1).collect()
if not approvals:
    blockers.append('HUMAN_APPROVAL_REQUIRED_FOR_EXACT_REPORT_VERSION')

if action == 'EXPORT' and not environment['export_enabled']:
    blockers.append('EXPORT_DISABLED_FOR_ENVIRONMENT')
if action == 'TRANSMIT':
    interface = requirement['official_interface']
    if not environment['transmission_enabled']:
        blockers.append('TRANSMISSION_DISABLED_FOR_ENVIRONMENT')
    if not interface['documented'] or not interface['reference'] or 'TODO' in interface['reference'].upper():
        blockers.append('OFFICIAL_INTERFACE_NOT_DOCUMENTED')
    if not interface['transmission_authorized'] or not interface['authorization_reference'] or 'TODO' in interface['authorization_reference'].upper():
        blockers.append('TRANSMISSION_NOT_AUTHORIZED')

allowed = not blockers
status = ('READY_FOR_MANUAL_EXPORT' if action == 'EXPORT' else 'READY_FOR_AUTHORIZED_TRANSMISSION') if allowed else 'BLOCKED'

In [ ]:
action_at = datetime.now(timezone.utc)
evidence = {
    'submission_id': submission_id,
    'report_version_hash': submission['report_version_hash'],
    'action': action,
    'action_status': status,
    'actor_object_id': actor_object_id,
    'blockers': blockers,
    'evaluated_at_utc': action_at.isoformat(),
    'automatic_authority_call_performed': False,
}
evidence_text = json.dumps(evidence, sort_keys=True)
evidence_sha256 = hashlib.sha256(evidence_text.encode()).hexdigest()
action_event = [(
    'EASA-ACTION-' + uuid.uuid4().hex, submission_id, submission['report_version_hash'],
    action, status, actor_object_id, action_at, json.dumps(blockers), evidence_sha256, submission['is_synthetic'])]
action_schema = 'action_event_id string, submission_id string, report_version_hash string, action_type string, action_status string, actor_object_id string, action_at_utc timestamp, blocker_codes_json string, evidence_sha256 string, is_synthetic boolean'
spark.createDataFrame(action_event, action_schema).write.mode('append').format('delta').saveAsTable('gold_easa_action_event')
notebookutils.fs.put(evidence_root + '/release-' + submission_id + '-' + evidence_sha256 + '.json', json.dumps(evidence, indent=2), True)
print(json.dumps(evidence, indent=2))
assert allowed, 'RELEASE_BLOCKED: ' + ','.join(blockers)
print(status + ': no authority endpoint was called')